# 01 - Carga de datos

En este notebook simplemente cargamos los datos 

In [19]:
import pyarrow.dataset as ds
import pyarrow.fs as fs
import pyarrow.parquet as pq


In [24]:
# Busqueda robusta del parquet usando solo pyarrow
sistema_archivos_local = fs.LocalFileSystem()

rutas_candidatas = [
    r"C:\Users\alons\Desktop\Práctica 7\sp500_history.parquet",
]

# Elimina duplicados conservando orden
rutas_unicas = []
for ruta in rutas_candidatas:
    if ruta not in rutas_unicas:
        rutas_unicas.append(ruta)

def es_parquet_valido(ruta):
    try:
        info = sistema_archivos_local.get_file_info(ruta)
        if info.type != fs.FileType.File:
            return False
        if info.size is None or info.size <= 100:
            return False
        _ = pq.ParquetFile(ruta).metadata
        return True
    except Exception:
        return False

PARQUET_PATH = next((ruta for ruta in rutas_unicas if es_parquet_valido(ruta)), None)

if PARQUET_PATH is None:
    raise FileNotFoundError(
        "No se encontro un parquet valido. Ajusta PARQUET_PATH manualmente o coloca sp500_history.parquet en data/raw/."
    )

print("PARQUET_PATH detectado:", PARQUET_PATH)


PARQUET_PATH detectado: C:\Users\alons\Desktop\Práctica 7\sp500_history.parquet


In [25]:
pf = pq.ParquetFile(PARQUET_PATH)
schema = pf.schema_arrow

print("Filas totales:", pf.metadata.num_rows)
print("Row groups:", pf.metadata.num_row_groups)
print("\nColumnas:", schema.names)

print("\nTipos:")
for c in schema.names:
    print(f"{c} -> {schema.field(c).type}")


Filas totales: 7250110
Row groups: 1289

Columnas: ['date', 'symbol', 'assetid', 'security_name', 'sector', 'industry', 'subsector', 'in_sp500', 'open', 'high', 'low', 'close', 'volume', 'unadjusted_close']

Tipos:
date -> timestamp[ns]
symbol -> string
assetid -> int64
security_name -> string
sector -> string
industry -> string
subsector -> string
in_sp500 -> int32
open -> float
high -> float
low -> float
close -> float
volume -> float
unadjusted_close -> float


In [26]:
# Intentamos mostrar columnas tÃ­picas si existen; si no, primeras 8
candidate = ["date","symbol","ticker","open","close","adj_close","unadjusted_close","volume","in_sp500"]
cols = schema.names
preview_cols = [c for c in candidate if c in cols]
if len(preview_cols) == 0:
    preview_cols = cols[:8]

parquet = pf.read_row_group(0, columns=preview_cols).to_pandas().head(10)
print(parquet)


        date symbol       open      close  unadjusted_close        volume  \
0 1999-11-18      A  27.188307  25.545057           42.7500  7.486229e+07   
1 1999-11-19      A  25.657097  24.349968           40.7500  1.823611e+07   
2 1999-11-22      A  24.686087  26.067909           43.6250  7.874048e+06   
3 1999-11-23      A  25.395672  24.051195           40.2500  7.153099e+06   
4 1999-11-24      A  23.976501  24.536699           41.0625  5.797720e+06   
5 1999-11-26      A  24.424660  24.611393           41.1875  2.070304e+06   
6 1999-11-29      A  24.499353  25.171593           42.1250  4.877790e+06   
7 1999-11-30      A  25.096899  25.208939           42.1875  5.159442e+06   
8 1999-12-01      A  25.208939  25.657097           42.9375  3.540150e+06   
9 1999-12-02      A  26.142603  26.366682           44.1250  3.674868e+06   

   in_sp500  
0         0  
1         0  
2         0  
3         0  
4         0  
5         0  
6         0  
7         0  
8         0  
9         0 